# SciGeoGuard-X v11 - Confirmed EarthRefine repair experiment

This compact public notebook reproduces the v11 novelty layer from the repository code: named scientific refinement obligations, evidence certificates, conservative minimal repair, and re-verification.

**Evidence boundary:** the frozen 180-case labels and the ten metamorphic mutations are generated/author-controlled evidence. They measure regression consistency and repair behavior, not independent human accuracy.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from src.scigeoguard_v11 import (
    Artifact, Step, Intent, verify_science,
    evidence_certificate, synthesize_minimal_repair,
    run_metamorphic_suite,
)

ROOT = Path.cwd()
print('Repository root:', ROOT)


## 1. Ten-family metamorphic test
Every valid seed should remain `PROVEN`; each controlled semantic mutation should become `REFUTED`. The repair layer then searches for the smallest scientifically admissible edit and re-verifies the repaired workflow.


In [ ]:
rows, certificates = run_metamorphic_suite()
metamorphic = pd.DataFrame(rows)
display(metamorphic)

n = len(metamorphic)
detected = int(metamorphic['detected'].sum())
repaired = int((metamorphic['repair_status'] == 'REPAIRED').sum())
reverified = int(metamorphic['reverified_proven'].sum())
print({'families': n, 'detected': detected, 'repaired': repaired, 'reverified_proven': reverified})
assert n == 10 and detected == 10 and repaired == 9 and reverified == 9


## 2. Frozen 180-case repair experiment
This section uses the same 180-case frozen subset as the earlier baseline experiment. It checks that v11 preserves the frozen verifier verdicts and measures how many `REFUTED` workflows admit a conservative repair without inventing provenance or other evidence.


In [ ]:
cases = pd.read_csv(ROOT / 'results/history/v6/Blinded_180_cases.csv')
reference = pd.read_csv(ROOT / 'results/history/v6/Frozen_v3_reference.csv')
reference = reference[['case_id','generated_expected']]
cases = cases.merge(reference, on='case_id', how='left', validate='one_to_one')

records = []
obligations = []
for row in cases.itertuples(index=False):
    artifacts = [Artifact(**x) for x in json.loads(row.artifacts_json)]
    steps = [Step(**x) for x in json.loads(row.steps_json)]
    intent = Intent(**json.loads(row.workflow_intent))
    result = verify_science(artifacts, steps, intent)
    cert = evidence_certificate(artifacts, steps, intent, result=result, case_id=row.case_id)
    repair = synthesize_minimal_repair(artifacts, steps, intent, max_edits=2) if result['verdict'] == 'REFUTED' else None
    records.append({
        'case_id': row.case_id,
        'generated_expected': row.generated_expected,
        'verdict': result['verdict'],
        'matches_reference': result['verdict'] == row.generated_expected,
        'first_code': result['findings'][0]['code'] if result['findings'] else None,
        'repair_status': repair['status'] if repair else '',
        'repair_cost': repair.get('cost') if repair else None,
        'repair_reverified_proven': bool(repair and repair['status']=='REPAIRED' and repair['result']['verdict']=='PROVEN'),
    })
    obligations.extend({'case_id': row.case_id, **o} for o in cert['obligations'])

repair_df = pd.DataFrame(records)
obl_df = pd.DataFrame(obligations)
display(repair_df.head())

refuted = repair_df[repair_df.verdict == 'REFUTED']
safe = refuted[refuted.repair_status == 'REPAIRED']
summary = {
    'cases': len(repair_df),
    'reference_matches': int(repair_df.matches_reference.sum()),
    'refuted': len(refuted),
    'safe_repairs': len(safe),
    'safe_repair_rate': len(safe)/len(refuted),
    'reverified_proven': int(safe.repair_reverified_proven.sum()),
    'mean_successful_repair_cost': float(safe.repair_cost.mean()),
    'obligation_families': int(obl_df.obligation_id.nunique()),
    'diagnostic_codes': int(obl_df.code.nunique()),
}
print(json.dumps(summary, indent=2))
assert summary['reference_matches'] == 180
assert summary['refuted'] == 112
assert summary['safe_repairs'] == 47
assert summary['reverified_proven'] == 47
assert abs(summary['safe_repair_rate'] - 47/112) < 1e-12


## 3. Conservative refusal example
A vertical-datum mismatch is deliberately left unrepaired because changing a label would not perform a real datum transformation. This is a core design constraint: repair searches for scientifically admissible edits, not arbitrary changes that merely make a checker pass.


In [ ]:
vertical = metamorphic[metamorphic['family'] == 'vertical_datum_mismatch'].iloc[0]
print(vertical.to_dict())
assert vertical['mutated_verdict'] == 'REFUTED'
assert vertical['repair_status'] == 'NO_SAFE_REPAIR_FOUND'
